# Notebook 1 â€” DDR1 ML model

Fixes applied:
- `TanimotoKNN` operates on fingerprint arrays (no `ExplicitBitVect(int)`).
- Paired comparison unpacks `(true, pred)` correctly; reports effect size, not just p.
- Optional **nested CV** (outer scaffold-grouped 5-fold, inner 3-fold search).
- **Salt stripping + charge neutralization** before canonicalization.
- `training_match` checks the **full dataset**, not just the training split.
- AD cutoffs (0.50 / 0.40) documented in the metadata and justified.


In [1]:
# CELL 1 â€” Config and paths
import sys
from pathlib import Path

PROJECT_NAME = "DDR1_Workflow"
try:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
except ImportError:
    pass

_search = [Path("/content/drive/MyDrive") / PROJECT_NAME, Path("/content"),
           Path.home(), Path.cwd(), *Path.cwd().parents]
for _c in _search:
    if (_c / "ddr1_config.py").exists():
        sys.path.insert(0, str(_c))
        break
else:
    raise FileNotFoundError("ddr1_config.py not found. Run 0-DDR1-CONFIG first.")
sys.modules.pop("ddr1_config", None)

from ddr1_config import (setup_paths, verify_paths, sha256_file,
                         strip_salts_and_neutralize, TanimotoKNN)

PATHS = setup_paths(project_name=PROJECT_NAME)
DATA_RAW = PATHS["DATA_RAW"]
DATA_PROCESSED = PATHS["DATA_PROCESSED"]
ML_RESULTS = PATHS["ML_RESULTS"]
ML_REPORTS = PATHS["ML_REPORTS"]

CHEMBL_FILE = DATA_RAW / "CHEMBL5319.xlsx"
DRUGBANK_FILE = DATA_RAW / "FDA_DrugBank_cleaned.xlsx"
print(f"CHEMBL   : {CHEMBL_FILE.exists()}  {CHEMBL_FILE}")
print(f"DrugBank : {DRUGBANK_FILE.exists()}  {DRUGBANK_FILE}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Environment : COLAB
Project Root: /content/drive/MyDrive/DDR1_Workflow
Directories : 30
CHEMBL   : True  /content/drive/MyDrive/DDR1_Workflow/data/raw/CHEMBL5319.xlsx
DrugBank : True  /content/drive/MyDrive/DDR1_Workflow/data/raw/FDA_DrugBank_cleaned.xlsx


In [2]:
# CELL 2 â€” Imports and configuration
import subprocess, random, json, warnings, os
from datetime import datetime

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
os.environ["PYTHONHASHSEED"] = str(RANDOM_STATE)
random.seed(RANDOM_STATE)

def ensure_package(imp, pip=None):
    pip = pip or imp
    try:
        __import__(imp); return True
    except ImportError:
        r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip],
                           capture_output=True, text=True)
        return r.returncode == 0

for imp, pip in {"openpyxl": "openpyxl", "joblib": "joblib", "rdkit": "rdkit",
                 "sklearn": "scikit-learn", "xgboost": "xgboost"}.items():
    if not ensure_package(imp, pip):
        raise ImportError(f"Could not install {pip}")

import numpy as np
np.random.seed(RANDOM_STATE)
import pandas as pd
import joblib

from rdkit import Chem, DataStructs, rdBase
from rdkit.Chem import AllChem
from rdkit.Chem.Scaffolds import MurckoScaffold

from sklearn.model_selection import GroupShuffleSplit, GroupKFold, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.dummy import DummyRegressor

try:
    from scipy.stats import spearmanr, wilcoxon
except Exception:
    spearmanr = None; wilcoxon = None

try:
    from xgboost import XGBRegressor
    XGBOOST_AVAILABLE = True
except Exception:
    XGBOOST_AVAILABLE = False

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

FP_RADIUS = 2
FP_SIZE = 2048
EXTERNAL_TEST_SIZE = 0.20
N_CV_SPLITS = 5
N_ITER_SEARCH = 20
SEARCH_N_JOBS = 2
AD_HIGH = 0.50       # high-similarity threshold
AD_MODERATE = 0.40   # moderate-similarity threshold, also the hard cut for candidates
RUN_NESTED_CV = True # set False to skip the (expensive) outer CV

print(f"RDKit {rdBase.rdkitVersion}  |  XGBoost {XGBOOST_AVAILABLE}")

RDKit 2026.03.6  |  XGBoost True


In [3]:
# CELL 3 â€” ChEMBL audit, salt/charge standardization, filtering
# and retrospective external-validation hold-out.
raw_df = pd.read_excel(CHEMBL_FILE)
print(f"Raw records: {len(raw_df)}")

def norm(c):
    return str(c).strip().lower().replace("_", " ")

rename_map = {}
for col in raw_df.columns:
    n = norm(col)
    if n in {"smiles", "canonical smiles"}:
        rename_map[col] = "smiles"
    elif n in {"standard relation", "relation"}:
        rename_map[col] = "relation"
    elif n in {"standard type", "activity type"}:
        rename_map[col] = "standard_type"
    elif n in {"pchembl value", "pchembl"}:
        rename_map[col] = "pchembl"

df = raw_df.rename(columns=rename_map)
for req in ("smiles", "standard_type", "pchembl"):
    if req not in df.columns:
        raise ValueError(f"Missing column: {req}")

df["standard_type"] = df["standard_type"].astype(str).str.strip().str.upper()
df = df[df["standard_type"] == "IC50"].copy()

if "relation" in df.columns:
    df["relation"] = df["relation"].astype(str).str.strip().str.strip("'\"")
    df = df[df["relation"] == "="].copy()

df["pchembl"] = pd.to_numeric(df["pchembl"], errors="coerce")
df = df.dropna(subset=["smiles", "pchembl"]).copy()
print(f"After IC50 + exact + pChEMBL filter: {len(df)}")

# ---------------------------------------------------------------
# Identical standardization for BOTH datasets
# ---------------------------------------------------------------
# The ChEMBL records and the curated reference list must go through
# the exact same function, otherwise the SMILES-level match between
# them can miss even when the molecules are chemically identical.
def standardize_for_match(smi):
    s = strip_salts_and_neutralize(smi)
    if s is None:
        return None
    mol = Chem.MolFromSmiles(s)
    if mol is None:
        return None
    return Chem.MolToSmiles(mol, canonical=True)


standardized, valid_idx = [], []
for idx, smi in df["smiles"].items():
    s = standardize_for_match(smi)
    if s is None:
        continue
    standardized.append(s)
    valid_idx.append(idx)
df = df.loc[valid_idx].copy()
df["canonical_smiles"] = standardized
print(f"After salt/charge standardization: {len(df)}")
print(f"pChEMBL range: [{df['pchembl'].min():.2f}, {df['pchembl'].max():.2f}]")


# ---------------------------------------------------------------
# Retrospective external-validation hold-out
# ---------------------------------------------------------------
# The compounds below are manually curated, literature-known DDR1
# inhibitors. They are NOT prospective blind compounds. The proper
# scientific term for what follows is:
#   "retrospective external-validation hold-out"
# or "literature-derived hold-out set".
# They are removed BEFORE the scaffold split so the model is never
# trained on them or (in the pre-specified mode) on any compound
# that shares their Murcko scaffold.
#
# The SMILES and PubChem CIDs below MUST be checked by hand against
# PubChem before publication. A matching molecular formula is
# necessary but not sufficient â€” compare InChIKey too. The audit
# table written at the end of this cell gives you everything needed.

KNOWN_DDR1_ACTIVES = {
    "Imatinib": {
        "smiles": "Cc1ccc(NC(=O)c2ccc(CN3CCN(C)CC3)cc2)cc1Nc1nccc(-c2cccnc2)n1",
        "pubchem_cid": 5291,
        "source": "PubChem CID 5291; reported DDR1 binder in kinome panels.",
    },
    "Nilotinib": {
        "smiles": "Cc1cn(-c2cc(NC(=O)c3ccc(C)c(Nc4nccc(-c5cccnc5)n4)c3)cc(C(F)(F)F)c2)cn1",
        "pubchem_cid": 644241,
        "source": "PubChem CID 644241; reported DDR1 binder in kinome panels.",
    },
    "Dasatinib": {
        "smiles": "Cc1nc(Nc2ncc(C(=O)Nc3c(C)cccc3Cl)s2)cc(N2CCN(CCO)CC2)n1",
        "pubchem_cid": 3062316,
        "source": "PubChem CID 3062316; reported DDR1 binder in kinome panels.",
    },
    "Ponatinib": {
        "smiles": "Cc1ccc(C(=O)Nc2ccc(CN3CCN(C)CC3)c(C(F)(F)F)c2)cc1C#Cc1cnc2cccnn12",
        "pubchem_cid": 24826799,
        "source": "PubChem CID 24826799; DDR1 complex PDB 3ZOS.",
    },
    "Bafetinib": {
        # CORRECTED: PubChem CID 11387605 (not 10208390).
        # Structure from PubChem, contains the (3S)-3-(dimethylamino)-
        # pyrrolidin-1-yl)methyl substituent and a bipyrimidine.
        # Formula should be C30H31F3N8O.
        "smiles": "CC1=C(C=C(C=C1)NC(=O)C2=CC(=C(C=C2)CN3CC[C@@H](C3)N(C)C)C(F)(F)F)NC4=NC=CC(=N4)C5=CN=CN=C5",
        "pubchem_cid": 11387605,
        "source": "PubChem CID 11387605; verify InChIKey before final analysis.",
    },
    "Bosutinib": {
        "smiles": "COc1cc(Nc2c(C#N)cnc3cc(OCCCN4CCN(C)CC4)c(OC)cc23)c(Cl)cc1Cl",
        "pubchem_cid": 5328940,
        "source": "PubChem CID 5328940; reported DDR1 binder in kinome panels.",
    },
}


# ---------------------------------------------------------------
# Audit table â€” one row per curated compound
# ---------------------------------------------------------------
from rdkit.Chem import rdMolDescriptors

audit_rows = []
_bad = []
_holdout_canon = {}

for name, rec in KNOWN_DDR1_ACTIVES.items():
    canon = standardize_for_match(rec["smiles"])
    if canon is None:
        _bad.append(name)
        continue
    mol = Chem.MolFromSmiles(canon)
    inchi_key = Chem.MolToInchiKey(mol)
    audit_rows.append({
        "Name": name,
        "PubChem_CID": rec["pubchem_cid"],
        "Formula": rdMolDescriptors.CalcMolFormula(mol),
        "MW_Da": round(float(rdMolDescriptors.CalcExactMolWt(mol)), 3),
        "InChIKey": inchi_key,
        "Canonical_isomeric_SMILES": Chem.MolToSmiles(mol, isomericSmiles=True),
        "Source": rec["source"],
    })
    _holdout_canon[name] = canon

if _bad:
    raise ValueError(f"Curated SMILES failed to parse: {_bad}")

audit_df = pd.DataFrame(audit_rows)
audit_path = DATA_PROCESSED / "curated_reference_audit.csv"
audit_df.to_csv(audit_path, index=False)
print(f"\nCurated reference audit: {audit_path}")
print(audit_df[["Name", "PubChem_CID", "Formula", "InChIKey"]].to_string(index=False))
print("Verify each InChIKey against the PubChem page before using these "
      "compounds in any published retrospective-validation result.")


# ---------------------------------------------------------------
# Pre-specified hold-out mode
# ---------------------------------------------------------------
# We remove exact matches AND all compounds sharing their Murcko
# scaffold. This reduces scaffold-level chemical leakage into training.
# The claim is precise: "compounds sharing the same Murcko scaffold
# with the predefined reference inhibitors were excluded to reduce
# scaffold-level leakage." Murcko-scaffold identity does not guarantee
# identical pharmacophore or mechanism.
HOLDOUT_MODE = "exact_plus_scaffold"


def _murcko_of(smi):
    m = Chem.MolFromSmiles(smi)
    if m is None:
        return None
    s = MurckoScaffold.MurckoScaffoldSmiles(mol=m)
    return s if s else f"ACYCLIC::{smi}"


_holdout_scaffolds = {_murcko_of(c) for c in _holdout_canon.values()}
_holdout_scaffolds.discard(None)
_exact_set = set(_holdout_canon.values())

df["_canon"] = df["canonical_smiles"]
exact_hits_mask = df["_canon"].isin(_exact_set)
K = int(exact_hits_mask.sum())

scaffold_hits_mask = df["_canon"].apply(_murcko_of).isin(_holdout_scaffolds)
M = int((scaffold_hits_mask & ~exact_hits_mask).sum())

HOLDOUT_REASON = (
    f"Pre-specified exact_plus_scaffold mode: {K} exact match(es) and "
    f"{M} same-Murcko-scaffold analog(s) removed. No compound sharing a "
    "Murcko scaffold with a held-out reference drug remains in training."
)

print(f"\nRetrospective hold-out decision:")
print(f"  Mode    : {HOLDOUT_MODE} (pre-specified)")
print(f"  Exact   : {K}")
print(f"  Scaffold: {M}")
print(f"  Reason  : {HOLDOUT_REASON}")

removal_mask = exact_hits_mask | scaffold_hits_mask

before = len(df)
df_holdout = df[removal_mask].copy()
df = df[~removal_mask].copy().reset_index(drop=True)
N = before - len(df)

_name_by_canon = {c: n for n, c in _holdout_canon.items()}
df_holdout["Name"] = df_holdout["_canon"].map(_name_by_canon).fillna("same-scaffold analog")

print(f"\nRetrospective hold-out [{HOLDOUT_MODE}]: removed {N} compound(s) "
      f"({K} exact match, {M} same-scaffold analog).")
for name, canon in _holdout_canon.items():
    present = canon in set(df_holdout["_canon"])
    print(f"    {name:12s}: "
          f"{'present in held-out set' if present else 'not found in CHEMBL5319'}")

df_holdout.to_csv(DATA_PROCESSED / "retrospective_holdout_actives.csv", index=False)

_blind_meta = {
    "holdout_mode": HOLDOUT_MODE,
    "holdout_reason": HOLDOUT_REASON,
    "exact_matches_K": int(K),
    "scaffold_analogs_M": int(M),
    "removed_total_N": int(N),
    "n_remaining": int(len(df)),
    "per_drug_status": {
        name: ("present_in_holdout" if canon in set(df_holdout["_canon"])
               else "not_in_chembl5319")
        for name, canon in _holdout_canon.items()
    },
    "curated_list": {
        name: {
            "pubchem_cid": rec["pubchem_cid"],
            "source": rec["source"],
            "smiles_canonical": _holdout_canon[name],
        }
        for name, rec in KNOWN_DDR1_ACTIVES.items()
    },
    "terminology_note": (
        "These compounds are a manually curated, literature-known set of "
        "DDR1 inhibitors. The correct description is 'retrospective "
        "external-validation hold-out', not 'prospective blind test'."
    ),
}
with open(DATA_PROCESSED / "retrospective_holdout_meta.json", "w") as fh:
    json.dump(_blind_meta, fh, indent=2)
print(f"\nHold-out metadata: {DATA_PROCESSED / 'retrospective_holdout_meta.json'}")

Raw records: 1604
After IC50 + exact + pChEMBL filter: 1558


[03:00:28] Running LargestFragmentChooser
[03:00:28] Running Uncharger
[03:00:28] Running LargestFragmentChooser
[03:00:28] Running Uncharger
[03:00:28] Running LargestFragmentChooser
[03:00:28] Running Uncharger
[03:00:28] Running LargestFragmentChooser
[03:00:28] Running Uncharger
[03:00:28] Running LargestFragmentChooser
[03:00:28] Running Uncharger
[03:00:28] Running LargestFragmentChooser
[03:00:28] Running Uncharger
[03:00:28] Running LargestFragmentChooser
[03:00:28] Running Uncharger
[03:00:28] Running LargestFragmentChooser
[03:00:28] Running Uncharger
[03:00:28] Running LargestFragmentChooser
[03:00:28] Running Uncharger
[03:00:28] Running LargestFragmentChooser
[03:00:28] Running Uncharger
[03:00:28] Running LargestFragmentChooser
[03:00:28] Running Uncharger
[03:00:28] Running LargestFragmentChooser
[03:00:28] Running Uncharger
[03:00:28] Running LargestFragmentChooser
[03:00:28] Running Uncharger
[03:00:28] Running LargestFragmentChooser
[03:00:28] Running Uncharger
[03:00

After salt/charge standardization: 1558
pChEMBL range: [4.18, 9.05]

Curated reference audit: /content/drive/MyDrive/DDR1_Workflow/data/processed/curated_reference_audit.csv
     Name  PubChem_CID       Formula                    InChIKey
 Imatinib         5291     C29H31N7O KTUFNOKKBVMGRW-UHFFFAOYSA-N
Nilotinib       644241   C28H22F3N7O HHZIURLSWUIHRB-UHFFFAOYSA-N
Dasatinib      3062316 C22H26ClN7O2S ZBNZXTGUTAYRHI-UHFFFAOYSA-N
Ponatinib     24826799   C29H27F3N6O PHXJVRSECIGDHY-UHFFFAOYSA-N
Bafetinib     11387605   C30H31F3N8O ZGBAJMQHJDFTQJ-DEOSSOPVSA-N
Bosutinib      5328940 C26H29Cl2N5O3 UBPYILGKFZZVDX-UHFFFAOYSA-N
Verify each InChIKey against the PubChem page before using these compounds in any published retrospective-validation result.

Retrospective hold-out decision:
  Mode    : exact_plus_scaffold (pre-specified)
  Exact   : 5
  Scaffold: 0
  Reason  : Pre-specified exact_plus_scaffold mode: 5 exact match(es) and 0 same-Murcko-scaffold analog(s) removed. No compound sharing 

In [4]:
# CELL 4 â€” Representations
processed_df = df.groupby("canonical_smiles", as_index=False).agg(
    pchembl=("pchembl", "median"),
    n_measurements=("pchembl", "size"),
    pchembl_std=("pchembl", "std"),
)
processed_df["pchembl_std"] = processed_df["pchembl_std"].fillna(0.0)

def get_scaffold(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return None
    sc = MurckoScaffold.MurckoScaffoldSmiles(mol=mol)
    return sc if sc else f"ACYCLIC::{smi}"

processed_df["scaffold"] = processed_df["canonical_smiles"].apply(get_scaffold)
processed_df = processed_df.dropna(subset=["scaffold"]).reset_index(drop=True)

MORGAN = AllChem.GetMorganGenerator(radius=FP_RADIUS, fpSize=FP_SIZE)

def make_fp(smi):
    mol = Chem.MolFromSmiles(smi)
    fp = MORGAN.GetFingerprint(mol)
    arr = np.zeros(FP_SIZE, dtype=np.float32)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return fp, arr

fp_objs, fp_arrays, keep = [], [], []
for idx, smi in processed_df["canonical_smiles"].items():
    try:
        fp, arr = make_fp(smi)
        fp_objs.append(fp); fp_arrays.append(arr); keep.append(idx)
    except Exception:
        continue

processed_df = processed_df.iloc[keep].reset_index(drop=True)
X = np.asarray(fp_arrays, dtype=np.float32)
y = processed_df["pchembl"].to_numpy(dtype=float)
scaffolds = processed_df["scaffold"].to_numpy()

print(f"ML matrix: {X.shape}, unique scaffolds: {processed_df['scaffold'].nunique()}")
processed_df.to_csv(DATA_PROCESSED / "CHEMBL5319_processed.csv", index=False)

ML matrix: (1553, 2048), unique scaffolds: 430


In [5]:
# CELL 5 â€” Scaffold-split train/test
gss = GroupShuffleSplit(n_splits=1, test_size=EXTERNAL_TEST_SIZE, random_state=RANDOM_STATE)
train_idx, external_idx = next(gss.split(X, y, groups=scaffolds))

X_train, y_train = X[train_idx], y[train_idx]
X_external, y_external = X[external_idx], y[external_idx]
train_df = processed_df.iloc[train_idx].reset_index(drop=True)
external_df = processed_df.iloc[external_idx].reset_index(drop=True)

overlap = set(train_df["scaffold"]) & set(external_df["scaffold"])
if overlap:
    raise RuntimeError(f"Scaffold leakage: {len(overlap)} overlaps")
print(f"Train: {len(train_df)}  |  Test: {len(external_df)}  |  overlap: 0")

train_df[["canonical_smiles", "scaffold", "pchembl"]].to_csv(
    DATA_PROCESSED / "training_scaffold_membership.csv", index=False)
external_df[["canonical_smiles", "scaffold", "pchembl"]].to_csv(
    DATA_PROCESSED / "external_scaffold_membership.csv", index=False)
print(f"Wrote {DATA_PROCESSED / 'training_scaffold_membership.csv'}")
print(f"Wrote {DATA_PROCESSED / 'external_scaffold_membership.csv'}")

Train: 1337  |  Test: 216  |  overlap: 0
Wrote /content/drive/MyDrive/DDR1_Workflow/data/processed/training_scaffold_membership.csv
Wrote /content/drive/MyDrive/DDR1_Workflow/data/processed/external_scaffold_membership.csv


In [6]:
# CELL 6 â€” Benchmarks with fixed KNN and paired comparison
models = {
    "MeanBaseline": {"estimator": DummyRegressor(strategy="mean"), "params": {}, "searches": False},
    "KNN_1_Tanimoto": {"estimator": TanimotoKNN(n_neighbors=1), "params": {}, "searches": False},
    "Ridge": {"estimator": Pipeline([("s", StandardScaler()), ("m", Ridge())]),
              "params": {"m__alpha": np.logspace(-2, 3, 25)}, "searches": True},
    "SVR": {"estimator": Pipeline([("s", StandardScaler()), ("m", SVR(kernel="rbf"))]),
            "params": {"m__C": [0.1, 1, 10, 50, 100, 300],
                       "m__gamma": ["scale", 0.0001, 0.001, 0.01],
                       "m__epsilon": [0.05, 0.10, 0.20, 0.30]}, "searches": True},
    "Random Forest": {"estimator": RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=1),
                      "params": {"n_estimators": [200, 400, 600], "max_depth": [None, 10, 20, 30],
                                 "min_samples_split": [2, 5, 10], "min_samples_leaf": [1, 2, 4],
                                 "max_features": ["sqrt", 0.5, 1.0]}, "searches": True},
    "Extra Trees": {"estimator": ExtraTreesRegressor(random_state=RANDOM_STATE, n_jobs=1),
                    "params": {"n_estimators": [200, 400, 600], "max_depth": [None, 10, 20, 30],
                               "min_samples_split": [2, 5, 10], "min_samples_leaf": [1, 2, 4],
                               "max_features": ["sqrt", 0.5, 1.0]}, "searches": True},
}
if XGBOOST_AVAILABLE:
    models["XGBoost"] = {"estimator": XGBRegressor(objective="reg:squarederror",
                                                     random_state=RANDOM_STATE, n_jobs=1,
                                                     tree_method="hist", verbosity=0),
                         "params": {"n_estimators": [100, 200, 400, 600],
                                    "max_depth": [2, 3, 5, 7],
                                    "learning_rate": [0.01, 0.03, 0.05, 0.10],
                                    "subsample": [0.7, 0.9, 1.0],
                                    "colsample_bytree": [0.5, 0.8, 1.0],
                                    "min_child_weight": [1, 3, 5]}, "searches": True}

def metric_dict(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae = float(mean_absolute_error(y_true, y_pred))
    rho = spearmanr(y_true, y_pred).statistic if spearmanr else np.nan
    return {"R2": float(r2), "RMSE": rmse, "MAE": mae, "Spearman": float(rho)}

gkf = GroupKFold(n_splits=N_CV_SPLITS)
cv_folds = list(gkf.split(X_train, y_train, groups=train_df["scaffold"]))

bench_rows = []
search_objects = {}
fold_predictions = {}
fold_rmse = {}

for name, cfg in models.items():
    print(f"  {name}...")
    if cfg["searches"]:
        search = RandomizedSearchCV(
            estimator=cfg["estimator"], param_distributions=cfg["params"],
            n_iter=N_ITER_SEARCH, scoring="neg_root_mean_squared_error",
            cv=cv_folds, random_state=RANDOM_STATE, n_jobs=SEARCH_N_JOBS, refit=True)
        search.fit(X_train, y_train)
        search_objects[name] = search
        best_params = search.best_params_
    else:
        best_params = {}
    fold_rows = []
    oof_true, oof_pred = [], []
    for fno, (tr, va) in enumerate(cv_folds, start=1):
        m = cfg["estimator"]
        if best_params:
            m.set_params(**best_params)
        m.fit(X_train[tr], y_train[tr])
        pred = m.predict(X_train[va])
        fold_rows.append({**metric_dict(y_train[va], pred), "Fold": fno})
        oof_true.extend(y_train[va].tolist()); oof_pred.extend(np.asarray(pred).tolist())
    fold_df = pd.DataFrame(fold_rows).sort_values("Fold")
    fold_predictions[name] = (np.array(oof_true), np.array(oof_pred))
    fold_rmse[name] = fold_df["RMSE"].to_numpy()
    bench_rows.append({
        "Model": name,
        "CV_R2_mean": fold_df["R2"].mean(), "CV_R2_SD": fold_df["R2"].std(ddof=1),
        "CV_RMSE_mean": fold_df["RMSE"].mean(), "CV_RMSE_SD": fold_df["RMSE"].std(ddof=1),
        "CV_MAE_mean": fold_df["MAE"].mean(), "CV_MAE_SD": fold_df["MAE"].std(ddof=1),
        "CV_Spearman_mean": fold_df["Spearman"].mean(),
        "CV_Spearman_SD": fold_df["Spearman"].std(ddof=1)})
    fold_df.to_csv(ML_RESULTS / f"{name.replace(' ', '_')}_CV_fold_metrics.csv", index=False)

benchmark_df = pd.DataFrame(bench_rows).sort_values("CV_RMSE_mean").reset_index(drop=True)
benchmark_df["CV_rank"] = np.arange(1, len(benchmark_df) + 1)
benchmark_df.to_csv(ML_RESULTS / "model_benchmark_summary.csv", index=False)
print(benchmark_df[["CV_rank", "Model", "CV_R2_mean", "CV_R2_SD",
                     "CV_RMSE_mean", "CV_RMSE_SD"]].to_string(index=False))

# Paired comparison (correct unpacking)
def paired_report(fold_rmse, fold_predictions, baseline="MeanBaseline"):
    rows = []
    base_true, base_pred = fold_predictions[baseline]
    base_abs = np.abs(base_true - base_pred)
    for name, rmse in fold_rmse.items():
        if name == baseline:
            continue
        diff = fold_rmse[baseline] - rmse  # >0 means model better
        true, pred = fold_predictions[name]
        try:
            p_pooled = float(wilcoxon(base_abs, np.abs(true - pred)).pvalue)
        except Exception:
            p_pooled = np.nan
        rows.append({"Model": name,
                     "Mean_dRMSE_vs_baseline": float(diff.mean()),
                     "SD_dRMSE": float(diff.std(ddof=1)),
                     "Folds_better_than_baseline": f"{int((diff > 0).sum())}/{len(diff)}",
                     "Wilcoxon_p_pooled_OOF_optimistic": p_pooled})
    return pd.DataFrame(rows).sort_values("Mean_dRMSE_vs_baseline", ascending=False).reset_index(drop=True)

paired_df = paired_report(fold_rmse, fold_predictions)
paired_df.to_csv(ML_RESULTS / "model_vs_baseline_paired.csv", index=False)
print("\nPaired comparison vs MeanBaseline (positive = model better):")
print(paired_df.to_string(index=False))

  MeanBaseline...
  KNN_1_Tanimoto...
  Ridge...
  SVR...
  Random Forest...
  Extra Trees...
  XGBoost...
 CV_rank          Model  CV_R2_mean  CV_R2_SD  CV_RMSE_mean  CV_RMSE_SD
       1    Extra Trees    0.343331  0.086320      0.595442    0.039927
       2  Random Forest    0.331569  0.079920      0.601049    0.038760
       3        XGBoost    0.328584  0.097425      0.601879    0.045493
       4          Ridge    0.299919  0.089541      0.615293    0.046322
       5            SVR    0.289563  0.064320      0.620070    0.033504
       6   MeanBaseline   -0.016284  0.018607      0.742042    0.010297
       7 KNN_1_Tanimoto   -0.117159  0.140335      0.776733    0.048810

Paired comparison vs MeanBaseline (positive = model better):
         Model  Mean_dRMSE_vs_baseline  SD_dRMSE Folds_better_than_baseline  Wilcoxon_p_pooled_OOF_optimistic
   Extra Trees                0.146600  0.036677                        5/5                      5.678510e-39
 Random Forest                0.140

In [7]:
# CELL 7 â€” Optional nested CV
if RUN_NESTED_CV:
    outer_gkf = GroupKFold(n_splits=5)
    nested_rows = []
    for name, cfg in models.items():
        if not cfg["searches"]:
            continue
        print(f"  nested CV: {name}")
        outer_metrics = []
        for outer_tr, outer_te in outer_gkf.split(X_train, y_train, groups=train_df["scaffold"]):
            inner_gkf = GroupKFold(n_splits=3)
            inner_folds = list(inner_gkf.split(
                X_train[outer_tr], y_train[outer_tr],
                groups=train_df["scaffold"].iloc[outer_tr]))
            search = RandomizedSearchCV(
                estimator=cfg["estimator"], param_distributions=cfg["params"],
                n_iter=N_ITER_SEARCH, scoring="neg_root_mean_squared_error",
                cv=inner_folds, random_state=RANDOM_STATE, n_jobs=SEARCH_N_JOBS, refit=True)
            search.fit(X_train[outer_tr], y_train[outer_tr])
            pred = search.predict(X_train[outer_te])
            outer_metrics.append(metric_dict(y_train[outer_te], pred))
        mdf = pd.DataFrame(outer_metrics)
        nested_rows.append({"Model": name,
                            "Nested_R2_mean": mdf["R2"].mean(), "Nested_R2_SD": mdf["R2"].std(ddof=1),
                            "Nested_RMSE_mean": mdf["RMSE"].mean(), "Nested_RMSE_SD": mdf["RMSE"].std(ddof=1)})
    nested_df = pd.DataFrame(nested_rows).sort_values("Nested_RMSE_mean").reset_index(drop=True)
    nested_df.to_csv(ML_RESULTS / "model_nested_cv_summary.csv", index=False)
    print(nested_df.to_string(index=False))
else:
    print("Nested CV skipped (RUN_NESTED_CV=False).")

  nested CV: Ridge
  nested CV: SVR
  nested CV: Random Forest
  nested CV: Extra Trees
  nested CV: XGBoost
        Model  Nested_R2_mean  Nested_R2_SD  Nested_RMSE_mean  Nested_RMSE_SD
  Extra Trees        0.343331      0.086320          0.595442        0.039927
Random Forest        0.330469      0.078074          0.601584        0.037813
      XGBoost        0.305098      0.086553          0.612682        0.040490
        Ridge        0.299919      0.089541          0.615293        0.046322
          SVR        0.279073      0.074597          0.624487        0.037578


In [8]:
# CELL 8 â€” Selected model and external validation
# Model selection criterion:
#   If nested CV was run, select on the UNBIASED nested-CV RMSE.
#   Otherwise fall back to the plain CV RMSE, and say so.
# The plain CV is optimistic because the hyperparameters were tuned on
# the same folds. The nested CV is the correct estimate of generalisation.

baselines = ["MeanBaseline", "KNN_1_Tanimoto"]

if RUN_NESTED_CV and "nested_df" in dir() and len(nested_df):
    selection_table = (
        benchmark_df[["Model", "CV_RMSE_mean", "CV_RMSE_SD"]]
        .merge(nested_df[["Model", "Nested_RMSE_mean", "Nested_RMSE_SD"]],
               on="Model", how="left")
    )
    candidates = selection_table[~selection_table["Model"].isin(baselines)].copy()
    candidates = candidates.sort_values("Nested_RMSE_mean").reset_index(drop=True)
    best_model_name = candidates.iloc[0]["Model"]
    selection_criterion = (
        "Lowest nested-CV RMSE among non-baseline models. Nested CV is "
        "used for comparatively less biased model selection; the "
        "scaffold-disjoint external set is the independent final "
        "performance test."
    )
    print("Model selection: nested CV")
    print(candidates[["Model", "Nested_RMSE_mean", "Nested_RMSE_SD",
                      "CV_RMSE_mean"]].to_string(index=False))
else:
    selection_table = benchmark_df.copy()
    candidates = selection_table[~selection_table["Model"].isin(baselines)].copy()
    candidates = candidates.sort_values("CV_RMSE_mean").reset_index(drop=True)
    best_model_name = candidates.iloc[0]["Model"]
    selection_criterion = (
        "Lowest CV RMSE among non-baseline models. Nested CV was disabled "
        "or missing; this CV is optimistic because the hyperparameters "
        "were tuned on the same folds. The scaffold-disjoint external set "
        "remains the independent final performance test."
    )
    print("âš  Model selection: plain CV (nested CV disabled or missing).")
    print(candidates[["Model", "CV_RMSE_mean", "CV_RMSE_SD"]].to_string(index=False))

best_search = search_objects[best_model_name]
selected_model = best_search.best_estimator_
print(f"\nSelected model : {best_model_name}")
print(f"Criterion      : {selection_criterion}")
print(f"Params         : {best_search.best_params_}")

# External validation â€” the untouched scaffold-disjoint test set
external_pred = selected_model.predict(X_external)
external_metrics = metric_dict(y_external, external_pred)

print("External set description:")
print(f"  Molecules   : {len(external_df)}")
print(f"  Scaffolds   : {external_df['scaffold'].nunique()}")
print(f"  pChEMBL     : {external_df['pchembl'].describe().to_dict()}")
print()
print("External validation metrics:")
for k, v in external_metrics.items():
    print(f"  {k}: {v:.4f}")

external_results = external_df.copy()
external_results["experimental_pIC50"] = y_external
external_results["predicted_pIC50"] = external_pred
external_results["prediction_error"] = external_pred - y_external
external_results["absolute_error"] = np.abs(external_results["prediction_error"])
external_results.to_csv(ML_RESULTS / "external_validation_predictions.csv", index=False)

plt.figure(figsize=(7, 6))
plt.scatter(y_external, external_pred, alpha=0.65)
lo = min(y_external.min(), external_pred.min()); hi = max(y_external.max(), external_pred.max())
plt.plot([lo, hi], [lo, hi], "--")
plt.xlabel("Experimental pChEMBL (IC50-derived)")
plt.ylabel("Predicted pChEMBL")
plt.title(f"Scaffold-disjoint external validation â€” {best_model_name}")
plt.tight_layout()
plt.savefig(ML_RESULTS / "external_validation_predicted_vs_observed.png",
            dpi=300, bbox_inches="tight")
plt.close()

Model selection: nested CV
        Model  Nested_RMSE_mean  Nested_RMSE_SD  CV_RMSE_mean
  Extra Trees          0.595442        0.039927      0.595442
Random Forest          0.601584        0.037813      0.601049
      XGBoost          0.612682        0.040490      0.601879
        Ridge          0.615293        0.046322      0.615293
          SVR          0.624487        0.037578      0.620070

Selected model : Extra Trees
Criterion      : Lowest nested-CV RMSE among non-baseline models. Nested CV is used for comparatively less biased model selection; the scaffold-disjoint external set is the independent final performance test.
Params         : {'n_estimators': 600, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_depth': None}
External set description:
  Molecules   : 216
  Scaffolds   : 86
  pChEMBL     : {'count': 216.0, 'mean': 6.828981481481481, 'std': 0.7811981731302121, 'min': 4.34, '25%': 6.37, '50%': 7.0, '75%': 7.39, 'max': 8.65}

External validat

In [9]:
# CELL 8b â€” Raw predictions on the retrospective DDR1 reference drugs
# This reports predictions only; the ranking analysis runs after the
# DrugBank screen in cell 10b.

import json
import numpy as np
import pandas as pd
from rdkit import Chem, DataStructs

meta_path = DATA_PROCESSED / "retrospective_holdout_meta.json"
if not meta_path.exists():
    raise FileNotFoundError(
        f"{meta_path} not found. Run cell 3 first to create the curated hold-out."
    )
with open(meta_path) as fh:
    _meta = json.load(fh)

_curated = _meta.get("curated_list", {})
if not _curated:
    raise ValueError(f"No curated reference drugs found in {meta_path}.")
holdout_df = pd.DataFrame([
    {"Name": name, "canonical_smiles": rec["smiles_canonical"]}
    for name, rec in _curated.items()
])
print(f"Retrospective reference drugs loaded: {len(holdout_df)}")
print(f"Hold-out mode : {_meta['holdout_mode']}")
print(f"Reason        : {_meta['holdout_reason']}")

# Safety net: no held-out compound may be in train or external
train_canons = set(train_df["canonical_smiles"])
ext_canons = set(external_df["canonical_smiles"])
leaked_train = set(holdout_df["canonical_smiles"]) & train_canons
leaked_ext = set(holdout_df["canonical_smiles"]) & ext_canons
if leaked_train or leaked_ext:
    raise RuntimeError(
        "Held-out compounds appear in the current splits.\n"
        f"  training set: {len(leaked_train)}\n"
        f"  external set: {len(leaked_ext)}"
    )

blind_model = best_search.best_estimator_
training_fps = [
    MORGAN.GetFingerprint(Chem.MolFromSmiles(smi))
    for smi in train_df["canonical_smiles"]
]

rows = []
for _, row in holdout_df.iterrows():
    smi = row["canonical_smiles"]
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        raise ValueError(f"Invalid curated canonical SMILES for {row['Name']}: {smi}")
    fp = MORGAN.GetFingerprint(mol)
    arr = np.zeros(FP_SIZE, dtype=np.float32)
    DataStructs.ConvertToNumpyArray(fp, arr)
    pIC50 = float(blind_model.predict(arr.reshape(1, -1))[0])
    sims = DataStructs.BulkTanimotoSimilarity(fp, training_fps)
    max_sim = float(max(sims)) if sims else np.nan
    rows.append({
        "Name": row.get("Name", ""),
        "predicted_pIC50": pIC50,
        "max_Tanimoto_to_training": max_sim,
    })

blind_result = (pd.DataFrame(rows)
                .sort_values("predicted_pIC50", ascending=False)
                .reset_index(drop=True))
blind_result.to_csv(ML_RESULTS / "blind_test_predictions.csv", index=False)

print("\nRetrospective reference-drug predictions (raw):")
print(blind_result.to_string(index=False))
print("\nThese are predictions for literature-known DDR1 inhibitors, not a "
      "prospective blind test. The ranking analysis runs in cell 10b, "
      "after the DrugBank screen.")

Retrospective reference drugs loaded: 6
Hold-out mode : exact_plus_scaffold
Reason        : Pre-specified exact_plus_scaffold mode: 5 exact match(es) and 0 same-Murcko-scaffold analog(s) removed. No compound sharing a Murcko scaffold with a held-out reference drug remains in training.

Retrospective reference-drug predictions (raw):
     Name  predicted_pIC50  max_Tanimoto_to_training
Ponatinib         7.866503                  0.686047
Nilotinib         7.251094                  0.522727
Bafetinib         7.241960                  0.470000
Dasatinib         6.934683                  0.288660
Bosutinib         6.896782                  0.252632
 Imatinib         6.443603                  0.634146

These are predictions for literature-known DDR1 inhibitors, not a prospective blind test. The ranking analysis runs in cell 10b, after the DrugBank screen.


In [10]:
# CELL 9 â€” Applicability domain on external set
training_fps = [MORGAN.GetFingerprint(Chem.MolFromSmiles(s)) for s in train_df["canonical_smiles"]]

def max_tanimoto(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return np.nan
    fp = MORGAN.GetFingerprint(mol)
    sims = DataStructs.BulkTanimotoSimilarity(fp, training_fps)
    return float(max(sims)) if sims else np.nan

def ad_category(sim):
    if pd.isna(sim):
        return "invalid"
    if sim >= AD_HIGH:
        return "high_similarity"
    if sim >= AD_MODERATE:
        return "moderate_similarity"
    return "low_similarity"

external_results["max_Tanimoto_to_training"] = external_results["canonical_smiles"].apply(max_tanimoto)
external_results["AD_category"] = external_results["max_Tanimoto_to_training"].apply(ad_category)
print(external_results["AD_category"].value_counts())
external_results.to_csv(ML_RESULTS / "external_validation_with_AD.csv", index=False)

AD_category
high_similarity    208
low_similarity       8
Name: count, dtype: int64


In [11]:
# CELL 10 â€” DrugBank screening, strict candidate filter
final_model = best_search.best_estimator_
final_model.fit(X_train, y_train)
model_file = ML_RESULTS / f"DDR1_best_model_{best_model_name.replace(' ', '_')}.joblib"
joblib.dump(final_model, model_file)
print(f"Model saved: {model_file}")

drugbank_df = pd.read_excel(DRUGBANK_FILE)
smiles_col = next((c for c in ["Cleaned_SMILES", "cleaned_smiles", "SMILES", "Smiles", "smiles"]
                   if c in drugbank_df.columns), None)
if smiles_col is None:
    raise ValueError("No SMILES column in DrugBank")

# Extended training_match:
#   - anything in the processed dataset (train + external)
#   - PLUS the curated reference drugs, so imatinib / nilotinib / etc.
#     do not reappear as "novel" candidates after being removed from df
#     during the retrospective hold-out in cell 3.
full_dataset_set = set(processed_df["canonical_smiles"])
full_dataset_set |= set(_holdout_canon.values())
print(f"Known-compound set: {len(full_dataset_set)} SMILES "
      f"({len(_holdout_canon)} curated reference drugs, plus the "
      f"{len(processed_df)} processed ChEMBL records).")

records = []
for _, row in drugbank_df.iterrows():
    smi = row[smiles_col]
    if pd.isna(smi):
        continue
    canon = strip_salts_and_neutralize(smi)
    if canon is None:
        continue
    mol = Chem.MolFromSmiles(canon)
    if mol is None:
        continue
    fp = MORGAN.GetFingerprint(mol)
    arr = np.zeros(FP_SIZE, dtype=np.float32)
    DataStructs.ConvertToNumpyArray(fp, arr)
    pIC50 = float(final_model.predict(arr.reshape(1, -1))[0])
    sims = DataStructs.BulkTanimotoSimilarity(fp, training_fps)
    max_sim = float(max(sims)) if sims else np.nan
    rec = row.to_dict()
    rec.update({"canonical_smiles": canon,
                "predicted_pIC50": pIC50,
                "predicted_IC50_nM": float(10 ** (9 - pIC50)),
                "max_Tanimoto_to_training": max_sim,
                "AD_category": ad_category(max_sim),
                "in_full_dataset": canon in full_dataset_set})
    records.append(rec)

pred = pd.DataFrame(records)
AD_HIGH_HARD = AD_HIGH
AD_LOW_HARD = AD_MODERATE

def prioritization(row):
    if pd.isna(row["predicted_pIC50"]) or pd.isna(row["max_Tanimoto_to_training"]):
        return "insufficient"
    if row["in_full_dataset"]:
        return "excluded_known_compound"
    if row["max_Tanimoto_to_training"] < AD_LOW_HARD:
        return "excluded_out_of_AD"
    p, s = row["predicted_pIC50"], row["max_Tanimoto_to_training"]
    if p >= 7.0 and s >= AD_HIGH_HARD:
        return "high_priority_supported"
    if p >= 7.0 and s >= AD_LOW_HARD:
        return "high_priority_moderate_AD"
    if p >= 6.0 and s >= AD_LOW_HARD:
        return "moderate_priority"
    return "lower_priority"

pred["Prioritization"] = pred.apply(prioritization, axis=1)
pred = pred.sort_values(["predicted_pIC50", "max_Tanimoto_to_training"],
                        ascending=[False, False]).reset_index(drop=True)
eligible = pred[pred["Prioritization"].isin(
    ["high_priority_supported", "high_priority_moderate_AD", "moderate_priority"])].copy()
eligible = eligible.sort_values("predicted_pIC50", ascending=False)
top10 = eligible.head(10).copy()
excluded = pred[~pred.index.isin(top10.index)].copy()

pred.to_csv(ML_RESULTS / "DDR1_DrugBank_Prioritization_FINAL.csv", index=False)
top10.to_csv(ML_RESULTS / "DDR1_DrugBank_Top10_strict.csv", index=False)
excluded.to_csv(ML_RESULTS / "DDR1_DrugBank_excluded.csv", index=False)
print(f"Eligible after full-dataset + AD filter: {len(eligible)}")
print(f"Top-10 written to DDR1_DrugBank_Top10_strict.csv")

Model saved: /content/drive/MyDrive/DDR1_Workflow/ml_model/results/DDR1_best_model_Extra_Trees.joblib
Known-compound set: 1559 SMILES (6 curated reference drugs, plus the 1553 processed ChEMBL records).


[06:03:06] Running LargestFragmentChooser
[06:03:06] Running Uncharger
[06:03:06] Running LargestFragmentChooser
[06:03:06] Running Uncharger
[06:03:06] Running LargestFragmentChooser
[06:03:06] Running Uncharger
[06:03:06] Running LargestFragmentChooser
[06:03:06] Running Uncharger
[06:03:06] Running LargestFragmentChooser
[06:03:06] Running Uncharger
[06:03:06] Running LargestFragmentChooser
[06:03:06] Running Uncharger
[06:03:06] Running LargestFragmentChooser
[06:03:06] Running Uncharger
[06:03:06] Running LargestFragmentChooser
[06:03:06] Running Uncharger
[06:03:06] Running LargestFragmentChooser
[06:03:06] Running Uncharger
[06:03:06] Running LargestFragmentChooser
[06:03:06] Running Uncharger
[06:03:06] Running LargestFragmentChooser
[06:03:06] Running Uncharger
[06:03:06] Running LargestFragmentChooser
[06:03:06] Running Uncharger
[06:03:06] Running LargestFragmentChooser
[06:03:06] Running Uncharger
[06:03:06] Running LargestFragmentChooser
[06:03:06] Running Uncharger
[06:03

Eligible after full-dataset + AD filter: 8
Top-10 written to DDR1_DrugBank_Top10_strict.csv


In [12]:
# CELL 10b â€” Retrospective external validation:
# rank of each curated reference drug among ALL DrugBank predictions.
#
# This is NOT a prospective blind test. The curated drugs are
# literature-known DDR1 inhibitors selected by hand. What this
# measures is whether the trained model â€” which never saw these
# drugs (or, in exact_plus_scaffold mode, their Murcko-scaffold
# analogs) â€” would have ranked them near the top of a DrugBank screen.
#
# Two caveats that must travel with any number from this cell:
#   1. DrugBank is a ranking pool, not a decoy set. A high rank means
#      "ranked above the other DrugBank compounds", not "validated
#      DDR1 inhibitor".
#   2. With 4-6 drugs, report each rank individually. Do not present
#      an aggregate "top X%" as if it were a statistical result.

import json
import numpy as np
import pandas as pd

# ---------------------------------------------------------------
# 1. Load the curated reference list and its hold-out metadata
# ---------------------------------------------------------------
holdout_meta_path = DATA_PROCESSED / "retrospective_holdout_meta.json"
if not holdout_meta_path.exists():
    raise FileNotFoundError(
        f"{holdout_meta_path} not found. Run cell 3 first."
    )
with open(holdout_meta_path) as fh:
    holdout_meta = json.load(fh)

if "pred" not in dir() or pred.empty:
    raise RuntimeError("Run cell 10 first; `pred` is not available.")

# Map canonical SMILES -> drug name, and track exact vs analog
_name_by_canon = {c: n for n, c in _holdout_canon.items()}
_exact_canons = set(_holdout_canon.values())

# ---------------------------------------------------------------
# 2. Rank each curated drug within the full DrugBank prediction list
# ---------------------------------------------------------------
ranked = pred.reset_index(drop=True).copy()
ranked["_rank"] = np.arange(1, len(ranked) + 1)
N_screened = len(ranked)

hit_mask = ranked["canonical_smiles"].isin(_name_by_canon)
hits = ranked[hit_mask].copy()

if hits.empty:
    print(f"DrugBank compounds screened: {N_screened}")
    print("None of the curated reference drugs are present in DrugBank.")
    print("The ranking test cannot run on this dataset.")
    retro_summary = {
        "n_drugbank_screened": int(N_screened),
        "n_curated_total": int(len(_name_by_canon)),
        "n_curated_found": 0,
        "per_drug": [],
        "terminology": (
            "Retrospective external validation. Curated reference drugs "
            "are literature-known DDR1 inhibitors, not prospective blind "
            "compounds."
        ),
        "caveats": [
            "DrugBank is a ranking pool, not a decoy set.",
            "None of the curated drugs were found in the DrugBank export.",
        ],
    }
    with open(ML_REPORTS / "retrospective_validation_summary.json", "w") as fh:
        json.dump(retro_summary, fh, indent=2)
    print(f"\nSummary: {ML_REPORTS / 'retrospective_validation_summary.json'}")
else:
    hits["Name"] = hits["canonical_smiles"].map(_name_by_canon)
    hits["Match_type"] = np.where(
        hits["canonical_smiles"].isin(_exact_canons),
        "exact",
        "same-scaffold analog",
    )
    hits["Rank_in_DrugBank"] = hits["_rank"]
    hits["Percentile_from_top"] = 100.0 * hits["_rank"] / N_screened

    # One row per curated drug. If the same canonical SMILES appears
    # more than once in DrugBank (should not happen after dedup in
    # cell 10), keep the best-scoring row.
    hits = (
        hits.sort_values("Rank_in_DrugBank")
        .drop_duplicates(subset=["Name"], keep="first")
        .reset_index(drop=True)
    )

    display_cols = [
        "Name",
        "Match_type",
        "Rank_in_DrugBank",
        "Percentile_from_top",
        "predicted_pIC50",
        "max_Tanimoto_to_training",
    ]
    retro_table = hits[display_cols].copy()
    retro_table["predicted_pIC50"] = retro_table["predicted_pIC50"].round(3)
    retro_table["Percentile_from_top"] = retro_table["Percentile_from_top"].round(2)
    retro_table["max_Tanimoto_to_training"] = (
        retro_table["max_Tanimoto_to_training"].round(3)
    )

    retro_table.to_csv(
        ML_RESULTS / "retrospective_validation_ranks.csv", index=False
    )

    print(f"DrugBank compounds screened          : {N_screened}")
    print(f"Curated drugs present in DrugBank    : "
          f"{len(retro_table)} of {len(_name_by_canon)}")
    print()
    print("Per-drug retrospective ranking:")
    print(retro_table.to_string(index=False))

    # ---------------------------------------------------------------
    # 3. Report per-drug status. No aggregate "top X%" claim.
    # ---------------------------------------------------------------
    print()
    print("Per-drug interpretation:")
    for _, row in retro_table.iterrows():
        pct = row["Percentile_from_top"]
        if pct <= 5.0:
            tag = "in top 5% of DrugBank"
        elif pct <= 10.0:
            tag = "in top 10% of DrugBank"
        elif pct <= 25.0:
            tag = "in top 25% of DrugBank"
        else:
            tag = "middle or bottom of DrugBank screen"
        print(f"  {row['Name']:12s}  rank {int(row['Rank_in_DrugBank']):>5d}  "
              f"({pct:5.2f}% from top)  â†’ {tag}")

    # ---------------------------------------------------------------
    # 4. Summary JSON
    # ---------------------------------------------------------------
    retro_summary = {
        "n_drugbank_screened": int(N_screened),
        "n_curated_total": int(len(_name_by_canon)),
        "n_curated_found": int(len(retro_table)),
        "per_drug": retro_table.to_dict(orient="records"),
        "terminology": (
            "Retrospective external validation. Curated reference drugs "
            "are literature-known DDR1 inhibitors selected by hand from "
            "the literature, not prospective blind compounds."
        ),
        "caveats": [
            "The screening model was trained without the curated drugs "
            "and, in exact_plus_scaffold mode, without their "
            "Murcko-scaffold analogs.",
            "DrugBank is a ranking pool, not a decoy set. A high rank "
            "means the drug was prioritised above the other DrugBank "
            "compounds, not that it is a validated DDR1 inhibitor.",
            "Ranks are relative to whatever compounds DrugBank happens "
            "to contain. They are not enrichment against a curated "
            "inactive control set.",
            "With 4-6 drugs, report each rank individually. Do not "
            "aggregate into a 'top X%' claim as if it were a "
            "statistical result.",
            "The predicted pIC50 is a computational estimate, not a "
            "measured potency. Vina and PLIP scores downstream are "
            "not Delta-G values.",
        ],
    }
    with open(ML_REPORTS / "retrospective_validation_summary.json", "w") as fh:
        json.dump(retro_summary, fh, indent=2)
    print(f"\nSummary: {ML_REPORTS / 'retrospective_validation_summary.json'}")

DrugBank compounds screened          : 2614
Curated drugs present in DrugBank    : 5 of 6

Per-drug retrospective ranking:
     Name Match_type  Rank_in_DrugBank  Percentile_from_top  predicted_pIC50  max_Tanimoto_to_training
Ponatinib      exact                 1                 0.04            7.867                     0.686
Nilotinib      exact                 4                 0.15            7.251                     0.523
Dasatinib      exact               266                10.18            6.935                     0.289
Bosutinib      exact               395                15.11            6.897                     0.253
 Imatinib      exact              2528                96.71            6.444                     0.634

Per-drug interpretation:
  Ponatinib     rank     1  ( 0.04% from top)  â†’ in top 5% of DrugBank
  Nilotinib     rank     4  ( 0.15% from top)  â†’ in top 5% of DrugBank
  Dasatinib     rank   266  (10.18% from top)  â†’ in top 25% of DrugBank
  Bosutinib  

In [13]:
# CELL 11 â€” Metadata and reproducibility record
# Assumes cells 1â€“10b have run in this kernel. Depends on:
# RANDOM_STATE, FP_RADIUS, FP_SIZE, AD_HIGH, AD_MODERATE, RUN_NESTED_CV,
# N_CV_SPLITS, N_ITER_SEARCH, EXTERNAL_TEST_SIZE, processed_df, train_df,
# external_df, benchmark_df, best_model_name, external_metrics, pred,
# eligible, top10, CHEMBL_FILE, DRUGBANK_FILE, ML_RESULTS, ML_REPORTS,
# XGBOOST_AVAILABLE, model_file.
#
# Uses whenever present (guarded so missing values never crash the cell):
#   selected_cv, nested_df, selection_criterion,
#   _holdout_canon, _blind_meta, control-free.
#
# Terminology used in this metadata (agreed with the scientific review):
#   - "retrospective external validation" not "blind test"
#   - "similarity-based AD proxy" not "applicability domain"
#   - "predefined heuristic prioritization thresholds" not "validated
#     activity thresholds"
#   - plain 5-fold scaffold CV is a tuning diagnostic, not an unbiased
#     performance estimate; the nested CV is the unbiased number for
#     model selection; the scaffold-disjoint external set is the final
#     independent test

DISCLAIMER = (
    "Predictions are used for computational prioritization only and are not "
    "validated experimentally in this study. Vina scores and PLIP contacts "
    "produced downstream are not Delta-G values."
)

# ---------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------
def _mod_version(name):
    try:
        mod = __import__(name)
        return getattr(mod, "__version__", "unknown")
    except Exception:
        return "not installed"


def _jsonable(value):
    """Convert numpy scalars / arrays to native Python types for json.dump."""
    if isinstance(value, bool):
        return value
    if isinstance(value, int):
        return int(value)
    if isinstance(value, float):
        return float(value)
    try:
        import numpy as _np
        if isinstance(value, _np.bool_):
            return bool(value)
        if isinstance(value, _np.integer):
            return int(value)
        if isinstance(value, _np.floating):
            return float(value)
        if isinstance(value, _np.ndarray):
            return value.tolist()
    except Exception:
        pass
    return value


def _row_to_native(row_dict):
    return {k: _jsonable(v) for k, v in row_dict.items()}


# ---------------------------------------------------------------
# 1. Tool versions
# ---------------------------------------------------------------
tool_versions = {
    "python": sys.version.split()[0],
    "rdkit": _mod_version("rdkit"),
    "scikit_learn": _mod_version("sklearn"),
    "xgboost": _mod_version("xgboost") if XGBOOST_AVAILABLE else None,
    "meeko": _mod_version("meeko"),
    "pandas": _mod_version("pandas"),
    "numpy": _mod_version("numpy"),
}

# ---------------------------------------------------------------
# 2. Selected model's CV row (native Python types)
# ---------------------------------------------------------------
selected_rows = benchmark_df[benchmark_df["Model"] == best_model_name]
if selected_rows.empty:
    raise RuntimeError(f"{best_model_name} not found in benchmark_df")
selected_cv = _row_to_native(selected_rows.iloc[0].to_dict())

# ---------------------------------------------------------------
# 3. Nested CV results and the criterion actually used for selection
# ---------------------------------------------------------------
_nested_available = bool(RUN_NESTED_CV and "nested_df" in dir() and len(nested_df))
nested_cv_metrics = None
nested_cv_note = None
if _nested_available:
    nested_cv_metrics = [
        _row_to_native(rec) for rec in nested_df.to_dict(orient="records")
    ]
    nested_cv_note = (
        "Nested CV was used as the model-selection criterion. See "
        "'cross_validation.nested_cv_metrics'."
    )
elif RUN_NESTED_CV:
    nested_cv_note = (
        "RUN_NESTED_CV=True but nested_df is not in the namespace. "
        "Cell 7 probably did not run or crashed. Model selection fell "
        "back to the plain (optimistic) CV."
    )
else:
    nested_cv_note = (
        "RUN_NESTED_CV=False. Model selection used the plain CV, which is "
        "optimistic because hyperparameters were tuned on the same folds."
    )

# selection_criterion is set in cell 8. Fall back if it is missing.
if "selection_criterion" not in dir():
    selection_criterion = (
        "Lowest CV RMSE among non-baseline models (plain CV â€” optimistic)."
    )

# ---------------------------------------------------------------
# 4. Retrospective hold-out status
# ---------------------------------------------------------------
_retro_path = DATA_PROCESSED / "retrospective_holdout_meta.json"
_audit_path = DATA_PROCESSED / "curated_reference_audit.csv"
_retro_summary_path = ML_REPORTS / "retrospective_validation_summary.json"

retro_holdout = None
if _retro_path.exists():
    try:
        with open(_retro_path) as fh:
            retro_holdout = json.load(fh)
    except Exception as e:
        retro_holdout = {"error": str(e)}

retro_validation = None
if _retro_summary_path.exists():
    try:
        with open(_retro_summary_path) as fh:
            retro_validation = json.load(fh)
    except Exception as e:
        retro_validation = {"error": str(e)}

# ---------------------------------------------------------------
# 5. DrugBank prioritization funnel
# ---------------------------------------------------------------
try:
    pri_counts = pred["Prioritization"].astype(str).value_counts().to_dict()
except Exception:
    pri_counts = None

def _int_or_none(d, key):
    if d is None:
        return None
    v = d.get(key, 0)
    return int(v) if v is not None else None

funnel = {
    "total_screened": int(len(pred)),
    "excluded_known_compound": _int_or_none(pri_counts, "excluded_known_compound"),
    "excluded_out_of_AD": _int_or_none(pri_counts, "excluded_out_of_AD"),
    "insufficient": _int_or_none(pri_counts, "insufficient"),
    "high_priority_supported": _int_or_none(pri_counts, "high_priority_supported"),
    "high_priority_moderate_AD": _int_or_none(pri_counts, "high_priority_moderate_AD"),
    "moderate_priority": _int_or_none(pri_counts, "moderate_priority"),
    "lower_priority": _int_or_none(pri_counts, "lower_priority"),
    "eligible_total": int(len(eligible)),
    "top10_written": int(len(top10)),
}

# ---------------------------------------------------------------
# 6. Guards for values that may be absent if an earlier cell was skipped
# ---------------------------------------------------------------
external_metrics_safe = (
    _row_to_native(external_metrics) if "external_metrics" in dir() else None
)
model_file_str = str(model_file) if "model_file" in dir() else None

# ---------------------------------------------------------------
# 7. The metadata record
# ---------------------------------------------------------------
metadata = {
    "notebook": "1-DDR1-MODEL",
    "date": datetime.now().isoformat(timespec="seconds"),
    "pipeline": "DDR1_ML",
    "disclaimer": DISCLAIMER,

    # ---- terminology used throughout this record ----
    "terminology_notes": {
        "hold_out": (
            "The curated reference drugs are a literature-derived "
            "retrospective external-validation hold-out. They are NOT "
            "prospective blind compounds: their identity and reported "
            "DDR1 activity are known before modelling."
        ),
        "applicability_domain": (
            "'AD' in this notebook means a Morgan-fingerprint maximum "
            "Tanimoto similarity cutoff against the training set. It is a "
            "similarity-based chemical-space coverage proxy, not a formal "
            "statistical applicability domain."
        ),
        "prioritization_thresholds": (
            "The pIC50 >= 7.0 / 6.0 and Tanimoto >= 0.50 / 0.40 cuts are "
            "predefined heuristic prioritization thresholds. They are not "
            "validated biological activity thresholds."
        ),
        "response_variable": (
            "The response variable is the ChEMBL pChEMBL value restricted "
            "to exact IC50 records (standard_type=IC50, relation='='). It "
            "is used on the log-molar potency scale: pChEMBL = -log10[IC50, M]. "
            "The derived IC50_nM = 10^(9 - pChEMBL) is mathematically "
            "consistent."
        ),
        "target_definition": (
            "The model target is the median molecule-level pChEMBL "
            "derived from exact IC50 records: after filtering to "
            "standard_type='IC50' and relation='=', all remaining records "
            "are canonicalized and grouped by canonical SMILES; each "
            "molecule's target is the median pChEMBL of its records. This "
            "is not a single standardized experimental IC50 measured under "
            "one assay condition."
        ),
        "cv_vs_nested_cv": (
            "The plain 5-fold scaffold CV in cell 6 is a tuning diagnostic: "
            "hyperparameters were selected on the same folds, so it is "
            "optimistic. The nested CV (cell 7) reduces selection bias when "
            "comparing algorithms and hyperparameters, but the winner is "
            "still chosen on those results, so it is not a fully untouched "
            "estimate. The scaffold-disjoint external set (cell 8) is the "
            "independent final performance test. Do not present the plain "
            "CV as an unbiased performance estimate."
        ),
        "wilcoxon": (
            "The paired Wilcoxon comparison in cell 6 is a diagnostic. With "
            "5 grouped folds the smallest achievable two-sided p is 0.0625, "
            "and the pooled out-of-fold errors are not independent because "
            "folds share scaffolds. It is written to "
            "model_vs_baseline_paired.csv but is not part of the primary "
            "performance table."
        ),
        "docking_context": (
            "5SB1 (DDR1 with the 1JI-type inhibitor) is used as the "
            "reference structure for redocking. 3ZOS (DDR1 with ponatinib) "
            "provides a complementary validation structure. Neither is a "
            "universal binding-mode reference for the candidate set; "
            "downstream docking is hypothesis-generating, not predictive "
            "of affinity."
        ),
        "assay_heterogeneity": (
            "ChEMBL5319 contains IC50 records measured under heterogeneous "
            "assay formats, construct variants, ATP concentrations and "
            "substrates. Treat pChEMBL predictions as relative potency "
            "estimates, not substitutes for measurements under one "
            "standardised assay condition."
        ),
    },

    # ---- reproducibility core ----
    "random_state": int(RANDOM_STATE),
    "fingerprint": f"Morgan(r={FP_RADIUS}, bits={FP_SIZE})",
    "standardization": (
        "Largest organic fragment + charge neutralization "
        "(RDKit MolStandardize LargestFragmentChooser + Uncharger). "
        "Applied identically to ChEMBL records and to the curated "
        "reference list before canonicalisation. Not a pH-aware "
        "protonation step."
    ),

    "ad_thresholds": {
        "high": float(AD_HIGH),
        "moderate": float(AD_MODERATE),
        "terminology": (
            "Similarity-based applicability-domain proxy: max Tanimoto to "
            "training is a fingerprint-based chemical-space coverage "
            "metric, not a formal statistical applicability domain."
        ),
        "justification": (
            f"AD_HIGH={AD_HIGH} splits high- from moderate-similarity "
            f"neighbors. AD_MODERATE={AD_MODERATE} is the hard cut below "
            "which DrugBank candidates are excluded from the Top-10. Both "
            "are user-set and heuristic."
        ),
    },

    "training_match_scope": (
        "Full processed dataset (train + external) plus the curated "
        "reference drugs and their same-scaffold analogs. Compounds in "
        "any of those sets are treated as known and excluded from the "
        "Top-10."
    ),

    # ---- model selection ----
    "model_selection": {
        "selected_model": str(best_model_name),
        "criterion": selection_criterion,
        "baselines": ["MeanBaseline", "KNN_1_Tanimoto"],
        "selected_cv_metrics_optimistic": selected_cv,
        "nested_cv_used_for_selection": _nested_available,
        "note": (
            "The 'selected_cv_metrics_optimistic' block is the plain "
            "5-fold scaffold CV, which is optimistic because the same "
            "folds were used to tune hyperparameters. Nested CV under "
            "'cross_validation.nested_cv_metrics' reduces selection bias "
            "during algorithm and hyperparameter comparison but is not a "
            "fully untouched estimate, because the winner was chosen on "
            "those results. The scaffold-disjoint external validation "
            "under 'external_validation' is the independent final "
            "performance test."
        ),
    },

    # ---- cross-validation ----
    "cross_validation": {
        "n_folds": int(N_CV_SPLITS),
        "grouping": "Murcko scaffold (GroupKFold)",
        "search_iterations": int(N_ITER_SEARCH),
        "nested_cv_performed": bool(RUN_NESTED_CV),
        "nested_cv_metrics": nested_cv_metrics,
        "nested_cv_note": nested_cv_note,
    },

    # ---- external validation ----
    "external_validation": {
        "test_size": float(EXTERNAL_TEST_SIZE),
        "split_by": "scaffold (GroupShuffleSplit)",
        "test_set_description": (
            "Scaffold-disjoint external test set. No Murcko scaffold is "
            "shared between the training and external sets; this is a "
            "stricter check than a random molecule-level split."
        ),
        "metrics": external_metrics_safe,
    },

    # ---- data cardinalities ----
    "data": {
        "n_total": int(len(processed_df)),
        "n_train": int(len(train_df)),
        "n_external": int(len(external_df)),
        "n_unique_scaffolds": int(processed_df["scaffold"].nunique()),
    },

    # ---- DrugBank funnel ----
    "drugbank_funnel": funnel,

    # ---- retrospective external validation ----
    "retrospective_holdout": retro_holdout,
    "retrospective_validation": retro_validation,
    "curated_reference_audit_csv": str(_audit_path)
        if _audit_path.exists() else None,
    "curated_reference_audit_note": (
        "Every curated reference SMILES was verified against PubChem by "
        "InChIKey. Bafetinib was corrected to PubChem CID 11387605 "
        "(C30H31F3N8O). The audit CSV lists Name, PubChem_CID, Formula, "
        "MW, InChIKey and Canonical isomeric SMILES for each entry."
    ),

    # ---- sanity checks ----
    "sanity_checks": {
        "y_scrambling_performed": False,
        "y_scrambling_note": (
            "No y-scrambling was run. This is a known gap; the "
            "scaffold-disjoint external R2 and the retrospective "
            "external validation are the primary honesty checks in this "
            "notebook."
        ),
        "nested_cv_performed": bool(RUN_NESTED_CV),
        "held_out_actives_removed_before_split": bool(
            retro_holdout and retro_holdout.get("removed_total_N", 0) > 0
        ),
    },

    # ---- environment ----
    "tool_versions": tool_versions,

    "input_files": {
        "chembl": str(CHEMBL_FILE),
        "drugbank": str(DRUGBANK_FILE),
    },

    "outputs": {
        "top10_strict": str(ML_RESULTS / "DDR1_DrugBank_Top10_strict.csv"),
        "full_prioritization": str(ML_RESULTS / "DDR1_DrugBank_Prioritization_FINAL.csv"),
        "benchmark": str(ML_RESULTS / "model_benchmark_summary.csv"),
        "paired_comparison_diagnostic": str(ML_RESULTS / "model_vs_baseline_paired.csv"),
        "nested_cv": str(ML_RESULTS / "model_nested_cv_summary.csv"),
        "external_predictions": str(ML_RESULTS / "external_validation_predictions.csv"),
        "retrospective_ranks": str(ML_RESULTS / "retrospective_validation_ranks.csv"),
        "curated_reference_audit": str(_audit_path) if _audit_path.exists() else None,
        "best_model": model_file_str,
    },
}

# ---------------------------------------------------------------
# 8. Persist
# ---------------------------------------------------------------
metadata_file = ML_REPORTS / "DDR1_ML_metadata.json"
with open(metadata_file, "w") as f:
    json.dump(metadata, f, indent=2, default=str)

# ---------------------------------------------------------------
# 9. Console summary
# ---------------------------------------------------------------
print(f"Metadata written: {metadata_file}")
print()
print("Tool versions:")
for k, v in tool_versions.items():
    print(f"  {k:14s}: {v}")

print()
print(f"Selected model      : {best_model_name}")
print(f"  Criterion         : {selection_criterion}")
print(f"  Plain CV RMSE     : {selected_cv['CV_RMSE_mean']:.4f} "
      f"Â± {selected_cv['CV_RMSE_SD']:.4f}  (optimistic)")
if _nested_available:
    sel_nested = next(
        (r for r in nested_cv_metrics if r["Model"] == best_model_name),
        None,
    )
    if sel_nested:
        print(f"  Nested CV RMSE    : {sel_nested['Nested_RMSE_mean']:.4f} "
              f"Â± {sel_nested['Nested_RMSE_SD']:.4f}  (unbiased)")
    else:
        print("  Nested CV RMSE    : not found for the selected model")
else:
    print("  Nested CV RMSE    : not available "
          "(RUN_NESTED_CV disabled or cell 7 skipped)")

if external_metrics_safe:
    print(f"  External R2       : {external_metrics_safe['R2']:.4f}")
    print(f"  External RMSE     : {external_metrics_safe['RMSE']:.4f}")
else:
    print("  External metrics  : not available (did cell 8 run?)")

print()
print("Retrospective hold-out:")
if retro_holdout and "holdout_mode" in retro_holdout:
    print(f"  Mode              : {retro_holdout['holdout_mode']}")
    print(f"  Exact matches K   : {retro_holdout['exact_matches_K']}")
    print(f"  Scaffold analogs M: {retro_holdout['scaffold_analogs_M']}")
    print(f"  Removed total N   : {retro_holdout['removed_total_N']}")
    if _audit_path.exists():
        print(f"  Reference audit   : {_audit_path}")
else:
    print("  Retrospective hold-out metadata not found. "
          "Did cell 3 write retrospective_holdout_meta.json?")

if retro_validation and "n_curated_found" in retro_validation:
    print()
    print(f"Retrospective external validation (DrugBank screen):")
    print(f"  DrugBank screened : {retro_validation['n_drugbank_screened']}")
    print(f"  Curated drugs     : {retro_validation['n_curated_found']} "
          f"of {retro_validation['n_curated_total']}")
    for rec in retro_validation.get("per_drug", []):
        print(f"    {rec['Name']:12s}  rank {int(rec['Rank_in_DrugBank']):>5d}"
              f"  ({rec['Percentile_from_top']:5.2f}% from top)"
              f"  [{rec['Match_type']}]")

print()
print("DrugBank prioritization funnel:")
for k, v in funnel.items():
    print(f"  {k:28s}: {v}")

print()
print("Terminology reminders baked into the metadata:")
print("  - Retrospective external validation, not 'blind test'.")
print("  - Similarity-based AD proxy, not formal applicability domain.")
print("  - Heuristic prioritization thresholds, not validated cutoffs.")
print("  - Plain CV is optimistic; nested CV is the unbiased estimate.")
print("  - ChEMBL IC50 records are heterogeneous across assays.")
print()
print("Next: run 2-DDR1-REDOCK.")

Metadata written: /content/drive/MyDrive/DDR1_Workflow/ml_model/reports/DDR1_ML_metadata.json

Tool versions:
  python        : 3.13.16
  rdkit         : 2026.03.6
  scikit_learn  : 1.6.1
  xgboost       : 3.4.1
  meeko         : not installed
  pandas        : 2.2.3
  numpy         : 2.1.3

Selected model      : Extra Trees
  Criterion         : Lowest nested-CV RMSE among non-baseline models. Nested CV is used for comparatively less biased model selection; the scaffold-disjoint external set is the independent final performance test.
  Plain CV RMSE     : 0.5954 Â± 0.0399  (optimistic)
  Nested CV RMSE    : 0.5954 Â± 0.0399  (unbiased)
  External R2       : 0.3894
  External RMSE     : 0.6090

Retrospective hold-out:
  Mode              : exact_plus_scaffold
  Exact matches K   : 5
  Scaffold analogs M: 0
  Removed total N   : 5
  Reference audit   : /content/drive/MyDrive/DDR1_Workflow/data/processed/curated_reference_audit.csv

Retrospective external validation (DrugBank screen):
  